## 1. Lenet 5 Architecture

In [9]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [10]:
x_train.shape

(60000, 28, 28)

In [11]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [15]:
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

In [16]:
x_train.shape

(60000, 28, 28, 1)

In [17]:
x_train = np.pad(x_train, ((0,0),(2,2),(2,2),(0,0)), 'constant')
x_test = np.pad(x_test, ((0,0),(2,2),(2,2),(0,0)), 'constant')

In [18]:
x_train.shape

(60000, 32, 32, 1)

In [21]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import numpy as np

# 1. Load Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 2. Preprocessing
# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape (add channel dimension)
x_train = x_train.reshape(-1, 28, 28, 1) # instead of we fill 60,000 images for batches, -1 says it to pick whatever we have fill it by yourself. In simple words, it is "auto-fill placeholder".
x_test = x_test.reshape(-1, 28, 28, 1)

# Pad to 32x32 (LeNet requirement)
x_train = np.pad(x_train, ((0,0),(2,2),(2,2),(0,0)), 'constant')
x_test = np.pad(x_test, ((0,0),(2,2),(2,2),(0,0)), 'constant')

# One-hot encoding
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 3. Build LeNet-5 Model
model = models.Sequential()

# C1 - Convolution
model.add(layers.Conv2D(6, kernel_size=(5,5), activation='tanh', input_shape=(32,32,1)))

# S2 - Average Pooling
model.add(layers.AveragePooling2D(pool_size=(2,2)))

# C3 - Convolution
model.add(layers.Conv2D(16, kernel_size=(5,5), activation='tanh'))

# S4 - Average Pooling
model.add(layers.AveragePooling2D(pool_size=(2,2)))

# C5 - Convolution (acts like fully connected)
model.add(layers.Conv2D(120, kernel_size=(5,5), activation='tanh'))

# Flatten
model.add(layers.Flatten())

# F6 - Fully Connected
model.add(layers.Dense(84, activation='tanh'))

# Output Layer
model.add(layers.Dense(10, activation='softmax'))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)                    │ (None, 28, 28, 6)           │             156 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ average_pooling2d_6                  │ (None, 14, 14, 6)           │               0 │
│ (AveragePooling2D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 10, 10, 16)          │           2,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ average_pooling2d_7                  │ (None, 5, 5, 16)            │               0 │
│ (AveragePooling2D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 1, 1, 120)           │          48,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 120)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 84)                  │          10,164 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 10)                  │             850 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# 4. Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train
model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test)
)

# 6. Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

# 7. Prediction Example
pred = model.predict(x_test[:5])
print("Predicted:", np.argmax(pred, axis=1))
print("Actual:", np.argmax(y_test[:5], axis=1))

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - accuracy: 0.9212 - loss: 0.2704 - val_accuracy: 0.9627 - val_loss: 0.1262
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.9696 - loss: 0.0994 - val_accuracy: 0.9756 - val_loss: 0.0774
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.9797 - loss: 0.0646 - val_accuracy: 0.9789 - val_loss: 0.0644
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.9855 - loss: 0.0474 - val_accuracy: 0.9839 - val_loss: 0.0531
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.9882 - loss: 0.0380 - val_accuracy: 0.9852 - val_loss: 0.0473
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9900 - loss: 0.0303 - val_accuracy: 0.9829 - val_loss: 0.0549
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.9922 - loss: 0.0246 - val_accuracy: 0.9844 - val_loss: 0.0489
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.9936 - loss: 0.0198 - 